# Importing the Dependencies

In [139]:
import numpy as np
import pandas as pd
import difflib # used to find the closest movie to the input
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Pre Processing

In [26]:
# loading the csv file to pandas dataframe
movie_data = pd.read_csv('movies.csv')

In [27]:
movie_data.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [28]:
movie_data.shape

(4803, 24)

In [29]:
movie_data.tail()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
4798,4798,220000,Action Crime Thriller,NaN,9367,united states\u2013mexico barrier legs arms pa...,es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,...,81.0,"[{""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238,Carlos Gallardo Jaime de Hoyos Peter Marquardt...,"[{'name': 'Robert Rodriguez', 'gender': 0, 'de...",Robert Rodriguez
4799,4799,9000,Comedy Romance,NaN,72766,NaN,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,...,85.0,[],Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5,Edward Burns Kerry Bish\u00e9 Marsha Dietlein ...,"[{'name': 'Edward Burns', 'gender': 2, 'depart...",Edward Burns
4800,4800,0,Comedy Drama Romance TV Movie,http://www.hallmarkchannel.com/signedsealeddel...,231617,date love at first sight narration investigati...,en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,...,120.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,"Signed, Sealed, Delivered",7.0,6,Eric Mabius Kristin Booth Crystal Lowe Geoff G...,"[{'name': 'Carla Hetland', 'gender': 0, 'depar...",Scott Smith
4801,4801,0,NaN,http://shanghaicalling.com/,126186,NaN,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,...,98.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,A New Yorker in Shanghai,Shanghai Calling,5.7,7,Daniel Henney Eliza Coupe Bill Paxton Alan Ruc...,"[{'name': 'Daniel Hsia', 'gender': 2, 'departm...",Daniel Hsia
4802,4802,0,Documentary,NaN,25975,obsession camcorder crush dream girl,en,My Date with Drew,Ever since the second grade when he first saw ...,1.929883,...,90.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,My Date with Drew,6.3,16,Drew Barrymore Brian Herzlinger Corey Feldman ...,"[{'name': 'Clark Peterson', 'gender': 2, 'depa...",Brian Herzlinger


In [30]:
movie_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [31]:
# checking for missing values
movie_data.isnull().sum()

index                      0
budget                     0
genres                    28
homepage                3091
id                         0
keywords                 412
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
cast                      43
crew                       0
director                  30
dtype: int64

## Feature Selection

In [32]:
# selecting the relevant features for recommendation

selected_features = ['genres','keywords','tagline','cast','director']
print(selected_features)

['genres', 'keywords', 'tagline', 'cast', 'director']


In [33]:
# Replacing the null values with null strings
for feature in selected_features:
    movie_data[feature] = movie_data[feature].fillna('')

In [34]:
# combining all the 5 selected features
combined_features = movie_data['genres']+' '+movie_data['keywords']+' '+movie_data['tagline']+' '+movie_data['cast']+' '+movie_data['director']

In [35]:
print(combined_features)

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      A New Yorker in Shanghai Daniel Henney Eliza...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object


In [36]:
# converting the text data to feature vectors
vectorizer = TfidfVectorizer()

In [37]:
feature_vectors = vectorizer.fit_transform(combined_features)

In [38]:
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 124266 stored elements and shape (4803, 17318)>
  Coords	Values
  (0, 201)	0.07860022416510505
  (0, 274)	0.09021200873707368
  (0, 5274)	0.11108562744414445
  (0, 13599)	0.1036413987316636
  (0, 5437)	0.1036413987316636
  (0, 3678)	0.21392179219912877
  (0, 3065)	0.22208377802661425
  (0, 5836)	0.1646750903586285
  (0, 14378)	0.33962752210959823
  (0, 16587)	0.12549432354918996
  (0, 3225)	0.24960162956997736
  (0, 14271)	0.21392179219912877
  (0, 4945)	0.24025852494110758
  (0, 15261)	0.07095833561276566
  (0, 16998)	0.1282126322850579
  (0, 11192)	0.09049319826481456
  (0, 11503)	0.27211310056983656
  (0, 13349)	0.15021264094167086
  (0, 17007)	0.23643326319898797
  (0, 17290)	0.20197912553916567
  (0, 13319)	0.2177470539412484
  (0, 14064)	0.20596090415084142
  (0, 16668)	0.19843263965100372
  (0, 14608)	0.15150672398763912
  (0, 8756)	0.22709015857011816
  :	:
  (4801, 403)	0.17727585190343229
  (4801, 4835)	0.247137650

## Cosine Similarity

In [39]:
# getting the similarity scores using cosine similarity

similarity = cosine_similarity(feature_vectors)

In [40]:
print(similarity)

[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]


In [41]:
similarity.shape

(4803, 4803)

## Taking the input movie name from User

In [147]:
movie_name = input("Enter your Favourite Movie Name: ")

Enter your Favourite Movie Name:  Conjuring


In [148]:
# Creating a list with all movies with names given in the dataset
list_of_all_titles = movie_data['title'].tolist()
print(list_of_all_titles)

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [149]:
# finding the close match for the movie name given by the user
find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
print(find_close_match)

['The Conjuring', 'The Conjuring 2', 'Counting']


In [150]:
close_match = find_close_match[0]
print(close_match)

The Conjuring


In [151]:
# finding the index of the movie with title

index_of_the_movie = movie_data[movie_data.title == close_match]['index'].values[0]
print(index_of_the_movie)

2096


## Getting list of similar Movies

In [152]:
# getting a list of similar movies

similarity_score = list(enumerate(similarity[index_of_the_movie])) # enumerate is used to carry oout a loop in our list
print(similarity_score)

[(0, np.float64(0.02666682916095678)), (1, np.float64(0.05902544175174934)), (2, np.float64(0.045072225664409805)), (3, np.float64(0.015216006182904786)), (4, np.float64(0.06450922804374952)), (5, np.float64(0.027624747228662663)), (6, np.float64(0.025543241519649377)), (7, np.float64(0.04446615395202721)), (8, np.float64(0.007279649074439151)), (9, np.float64(0.04344344948685305)), (10, np.float64(0.02206662637505684)), (11, np.float64(0.004406515980398065)), (12, np.float64(0.02810973848017647)), (13, np.float64(0.018085923752664573)), (14, np.float64(0.05237691180182792)), (15, np.float64(0.09769908511228437)), (16, np.float64(0.04216336611447135)), (17, np.float64(0.0)), (18, np.float64(0.00828071435722214)), (19, np.float64(0.021144222595813354)), (20, np.float64(0.03186752429931284)), (21, np.float64(0.0)), (22, np.float64(0.014193314510149598)), (23, np.float64(0.014988983133716926)), (24, np.float64(0.023002273689582978)), (25, np.float64(0.03531386567229851)), (26, np.float64(

In [153]:
len(similarity_score)

4803

In [154]:
# Sorting the movies based on hihger similarity score
sorted_similar_movies = sorted(similarity_score, key = lambda x:x[1], reverse= True)
print(sorted_similar_movies)

[(2096, np.float64(0.9999999999999999)), (1160, np.float64(0.34905754487883345)), (3270, np.float64(0.21622445704564244)), (4178, np.float64(0.18182048392695893)), (1590, np.float64(0.1672842780870894)), (3583, np.float64(0.15891062169318193)), (776, np.float64(0.15699238047549988)), (451, np.float64(0.15274943900571658)), (4415, np.float64(0.15240489508130897)), (2570, np.float64(0.1494639865389323)), (2004, np.float64(0.14417271985209257)), (3163, np.float64(0.1432175843313319)), (2904, np.float64(0.1424042058101468)), (310, np.float64(0.14090782675789126)), (3704, np.float64(0.1407210832700009)), (1745, np.float64(0.14066788229516902)), (1565, np.float64(0.14056258529823681)), (1291, np.float64(0.13776285958574253)), (4707, np.float64(0.13628481472776485)), (3937, np.float64(0.13575183131022267)), (1394, np.float64(0.13418099808952372)), (1901, np.float64(0.1318042108056738)), (4684, np.float64(0.12814601769065193)), (4717, np.float64(0.12453870708289086)), (3629, np.float64(0.12321

In [155]:
# print the name of similar movies based on the index
print('Movies Suggested for you: \n')

i=1

for movie in sorted_similar_movies:
    index = movie[0]
    title_from_index = movie_data[movie_data.index==index]['title'].values[0]
    if(i<=30):
        print(i,'.', title_from_index)
        i+=1

Movies Suggested for you: 

1 . The Conjuring
2 . The Conjuring 2
3 . Howards End
4 . Higher Ground
5 . Atonement
6 . Our Idiot Brother
7 . Winter's Tale
8 . The Haunting
9 . Snow White: A Deadly Summer
10 . Ramona and Beezus
11 . Marvin's Room
12 . Detention
13 . Heaven is for Real
14 . In the Heart of the Sea
15 . Salvador
16 . My Sister's Keeper
17 . Up in the Air
18 . The Time Traveler's Wife
19 . When the Lights Went Out
20 . Happiness
21 . Donnie Brasco
22 . The Greatest Game Ever Played
23 . American Beast
24 . Your Sister's Sister
25 . Bottle Shock
26 . Snitch
27 . The Immigrant
28 . 27 Dresses
29 . Won't Back Down
30 . The Ghost and the Darkness


## Movie Recommendation System

In [156]:
movie_name = input(' Enter your favourite movie name : ')

list_of_all_titles = movie_data['title'].tolist()

find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)

close_match = find_close_match[0]

index_of_the_movie = movie_data[movie_data.title == close_match]['index'].values[0]

similarity_score = list(enumerate(similarity[index_of_the_movie]))

sorted_similar_movies = sorted(similarity_score, key = lambda x:x[1], reverse = True) 

print('Movies suggested for you : \n')

i = 1

for movie in sorted_similar_movies:
  index = movie[0]
  title_from_index = movie_data[movie_data.index==index]['title'].values[0]
  if (i<30):
    print(i, '.',title_from_index)
    i+=1

 Enter your favourite movie name :  Conjuring


Movies suggested for you : 

1 . The Conjuring
2 . The Conjuring 2
3 . Howards End
4 . Higher Ground
5 . Atonement
6 . Our Idiot Brother
7 . Winter's Tale
8 . The Haunting
9 . Snow White: A Deadly Summer
10 . Ramona and Beezus
11 . Marvin's Room
12 . Detention
13 . Heaven is for Real
14 . In the Heart of the Sea
15 . Salvador
16 . My Sister's Keeper
17 . Up in the Air
18 . The Time Traveler's Wife
19 . When the Lights Went Out
20 . Happiness
21 . Donnie Brasco
22 . The Greatest Game Ever Played
23 . American Beast
24 . Your Sister's Sister
25 . Bottle Shock
26 . Snitch
27 . The Immigrant
28 . 27 Dresses
29 . Won't Back Down
